# Build Dataframe for All Metrics (RS_PRE=100, RS_POST=100) and Identify Top Pre-Metrics for N1/N2

This notebook:
1. Filters for RS_PRE=100 and RS_POST=100
2. Creates a DataFrame with ALL available metrics (100+)
3. Computes pre-post correlations across trials for each participant and session
4. Identifies which pre-metrics give the strongest correlations with N1, N1_t, N2, N2_t

In [1]:
from __future__ import annotations

# Standard library
import pickle
import re
from collections import defaultdict
from pathlib import Path
from itertools import product

# Third-party
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

In [2]:
# ============================================================================
# CONFIGURATION: Only RS_PRE=100 and RS_POST=100
# ============================================================================
RS_PRE_ALLOWED = {100}
RS_POST_ALLOWED = {100}

# Expected filename format, e.g sub-01_run-03_20_50_metrics.pkl
FILENAME_RE = re.compile(r'(sub-\d+)_run-(\d+)_(\d+)_(\d+)_metrics\.pkl$')

# Target metrics for final analysis
TARGET_POST_METRICS = {'N1', 'N1_t', 'N2', 'N2_t'}


def resolve_project_root() -> Path:
    """Resolve project root whether notebook is run from repo root or /notebooks."""
    cwd = Path.cwd().resolve()
    if (cwd / 'data').exists():
        return cwd
    if (cwd.parent / 'data').exists():
        return cwd.parent
    raise FileNotFoundError('Could not find project root containing data/.')


def normalize_metric_value(value):
    """Convert values to dataframe-friendly Python types.

    - np.ndarray(1) -> scalar
    - np.ndarray(n>1) -> list
    - list/tuple(1) -> single element
    - list/tuple(n>1) -> list
    - scalar -> unchanged
    """
    if isinstance(value, np.ndarray):
        if value.size == 1:
            return value.item()
        return value.tolist()
    if isinstance(value, (list, tuple)):
        if len(value) == 1:
            return value[0]
        return list(value)
    return value

In [3]:
def parse_file_all_metrics(data: dict, file_path: Path, records_map: dict):
    """Parse one pickle file and populate records_map with ALL metrics.
    
    Filters to RS_PRE=100 and RS_POST=100 only.
    """
    # Parse identifiers from filename
    filename_match = FILENAME_RE.search(file_path.name)
    if not filename_match:
        return

    sub_id = filename_match.group(1)
    run_id = f"run-{int(filename_match.group(2)):02d}"
    rpre_int = int(filename_match.group(3))
    rpost_int = int(filename_match.group(4))

    # Radius guard: only keep RS_PRE=100 and RS_POST=100
    if rpre_int not in RS_PRE_ALLOWED or rpost_int not in RS_POST_ALLOWED:
        return
    if not isinstance(data, dict):
        return

    # Pickle structure: metrics_pre/metrics_post -> metric_name -> trial_id -> value
    pre_root = data.get('metrics_pre', {})
    post_root = data.get('metrics_post', {})

    if not isinstance(pre_root, dict):
        pre_root = {}
    if not isinstance(post_root, dict):
        post_root = {}

    # Get ALL metric names available in this file
    all_metric_names = set()
    all_metric_names.update(pre_root.keys())
    all_metric_names.update(post_root.keys())

    # Process each metric
    for metric_name in all_metric_names:
        pre_trials = pre_root.get(metric_name, {}) if isinstance(pre_root, dict) else {}
        post_trials = post_root.get(metric_name, {}) if isinstance(post_root, dict) else {}

        # Merge trial IDs from pre and post
        trial_ids = set()
        if isinstance(pre_trials, dict):
            trial_ids.update(pre_trials.keys())
        if isinstance(post_trials, dict):
            trial_ids.update(post_trials.keys())

        for trial in trial_ids:
            try:
                trial_id = int(trial)
            except Exception:
                trial_id = trial

            key = (sub_id, rpre_int, rpost_int, run_id, trial_id, metric_name)
            rec = records_map[key]
            rec['sub'] = sub_id
            rec['radius_pre'] = rpre_int
            rec['radius_post'] = rpost_int
            rec['run_is'] = run_id
            rec['trial_id'] = trial_id
            rec['metric_name'] = metric_name

            if isinstance(pre_trials, dict) and trial in pre_trials:
                rec['metric_value_pre'] = normalize_metric_value(pre_trials[trial])
            if isinstance(post_trials, dict) and trial in post_trials:
                rec['metric_value_post'] = normalize_metric_value(post_trials[trial])

# Load iEEG Data with All Metrics (RS_PRE=100, RS_POST=100)

In [4]:
# Locate metrics folder
project_root = resolve_project_root()
metrics_root = project_root / 'data' / 'df_results' / 'ieeg_metrics'

# Collect all metric pickles
all_files = sorted(metrics_root.glob('sub-*/**/*_metrics.pkl'))
print(f'Found {len(all_files)} metric files in {metrics_root}')

# Aggregate parsed rows into a dict keyed by unique row dimensions
records_map = defaultdict(dict)

for pkl_file in all_files:
    with open(pkl_file, 'rb') as f:
        data = pickle.load(f)
    parse_file_all_metrics(data, pkl_file, records_map)

# Build dataframe from aggregated records
df_all_metrics = pd.DataFrame(records_map.values())

print(f'Total rows loaded: {len(df_all_metrics)}')
print(f'Unique metrics: {df_all_metrics["metric_name"].nunique()}')
print(f'Unique participants: {df_all_metrics["sub"].nunique()}')
print(f'Unique sessions: {df_all_metrics["run_is"].nunique()}')
print(f'\nMetric columns present: {sorted(df_all_metrics["metric_name"].unique())}')
print(f'\nDataframe shape: {df_all_metrics.shape}')
df_all_metrics.head(10)

Found 10494 metric files in /Users/cbc/Documents/GitHub/fufo/notebook/DavideMomi/Revision/State_Dependent_Brain_Stimulation-main/data/df_results/ieeg_metrics
Total rows loaded: 1409970
Unique metrics: 129
Unique participants: 36
Unique sessions: 17

Metric columns present: ['Dimensionality', 'Entropy_AEC', 'Entropy_COKU', 'Entropy_DFC', 'Entropy_FC', 'Entropy_PCA', 'Entropy_PCM', 'Entropy_PLV', 'Lempel-Ziv', 'Metastability', 'N1', 'N1_t', 'N2', 'N2_t', 'PCI', 'Power_Shannon_Entropy', 'Spectral Ratio', 'aec_matrix_avg_clustering_coefficient', 'aec_matrix_avg_eigenvector_centrality', 'aec_matrix_charpath_length', 'aec_matrix_coefficient_of_variation_mat', 'aec_matrix_first_moment_mat', 'aec_matrix_kurtosis_mat', 'aec_matrix_mean_mat', 'aec_matrix_mean_mean_mat', 'aec_matrix_mean_var_mat', 'aec_matrix_median_mat', 'aec_matrix_modularity', 'aec_matrix_norm', 'aec_matrix_skewness_mat', 'aec_matrix_var_mat', 'aec_matrix_var_mean_mat', 'aec_matrix_var_var_mat', 'co-kurtosis_matrix_avg_cluster

,sub,radius_pre,radius_post,run_is,trial_id,metric_name,metric_value_pre,metric_value_post
0,sub-01,100,100,run-01,0,mean_peak_to_peak,0.000155,0.000257
1,sub-01,100,100,run-01,1,mean_peak_to_peak,0.000086,0.000238
2,sub-01,100,100,run-01,2,mean_peak_to_peak,0.000091,0.000200
3,sub-01,100,100,run-01,3,mean_peak_to_peak,0.000125,0.000219
4,sub-01,100,100,run-01,4,mean_peak_to_peak,0.000080,0.000197
5,sub-01,100,100,run-01,5,mean_peak_to_peak,0.000082,0.000207
6,sub-01,100,100,run-01,6,mean_peak_to_peak,0.000173,0.000242
7,sub-01,100,100,run-01,7,mean_peak_to_peak,0.000117,0.000253
8,sub-01,100,100,run-01,8,mean_peak_to_peak,0.000145,0.000230
9,sub-01,100,100,run-01,9,mean_peak_to_peak,0.000171,0.000255


# Compute Pre-Post Correlations by Participant and Session

For each participant and session combination, compute correlations between pre and post values across all trials for every metric pair.

In [5]:
# Rename and structure data for correlation computation
df_corr_src = df_all_metrics.copy()
df_corr_src = df_corr_src.rename(columns={'radius_pre': 'Radius_pre', 'radius_post': 'Radius_post'})

# Patient label: sub-01 -> P1, sub-36 -> P36
df_corr_src['Patient'] = (
    df_corr_src['sub']
    .astype(str)
    .str.extract(r'(\d+)')[0]
    .astype(float)
    .astype('Int64')
    .astype(str)
    .radd('P')
)

# Session label from run order per patient
df_corr_src['run_num'] = (
    df_corr_src['run_is']
    .astype(str)
    .str.extract(r'(\d+)')[0]
    .astype(float)
)

session_map = (
    df_corr_src[['Patient', 'run_num']]
    .dropna()
    .drop_duplicates()
    .sort_values(['Patient', 'run_num'])
)
session_map['session_idx'] = session_map.groupby('Patient').cumcount() + 1

df_corr_src = df_corr_src.merge(session_map, on=['Patient', 'run_num'], how='left')
df_corr_src['Session'] = 'S' + df_corr_src['session_idx'].astype('Int64').astype(str)

print(f'Patients: {sorted(df_corr_src["Patient"].unique())}')
print(f'Sessions per patient (first patient): {df_corr_src[df_corr_src["Patient"] == df_corr_src["Patient"].iloc[0]]["Session"].nunique()}')
print(f'\nReady to compute correlations...')

Patients: ['P1', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P2', 'P20', 'P21', 'P22', 'P23', 'P24', 'P25', 'P26', 'P27', 'P28', 'P29', 'P3', 'P30', 'P31', 'P32', 'P33', 'P34', 'P35', 'P36', 'P4', 'P5', 'P6', 'P7', 'P8', 'P9']
Sessions per patient (first patient): 7

Ready to compute correlations...


In [7]:
# Compute correlations for all pre-metrics with target post-metrics only (N1, N1_t, N2, N2_t)
group_cols = ['Patient', 'Session', 'Radius_pre', 'Radius_post']
target_post_metrics = {'N1', 'N1_t', 'N2', 'N2_t'}

correlation_rows = []
total_groups = df_corr_src.groupby(group_cols, dropna=False).ngroups
processed = 0

for group_key, group_df in df_corr_src.groupby(group_cols, dropna=False):
    processed += 1
    if processed % 10 == 0:
        print(f'Processing group {processed}/{total_groups}...', end='\r')
    
    patient, session, radius_pre, radius_post = group_key

    # Pivot trial x metric matrices for pre and post values
    pre_wide = group_df.pivot_table(
        index='trial_id',
        columns='metric_name',
        values='metric_value_pre',
        aggfunc='mean'
    )
    post_wide = group_df.pivot_table(
        index='trial_id',
        columns='metric_name',
        values='metric_value_post',
        aggfunc='mean'
    )

    # Get all pre-metrics and filter post-metrics to target set
    all_pre_metrics = set(pre_wide.columns)
    available_post_metrics = set(post_wide.columns) & target_post_metrics

    # Compute correlations for all pre-metrics with target post-metrics only
    for pre_metric in sorted(all_pre_metrics):
        for post_metric in sorted(available_post_metrics):
            if pre_metric in pre_wide.columns and post_metric in post_wide.columns:
                pair_df = pd.concat(
                    [pre_wide[pre_metric], post_wide[post_metric]],
                    axis=1,
                    join='inner'
                ).dropna()
            else:
                pair_df = pd.DataFrame()

            # Compute Spearman correlation (requires at least 3 data points)
            if len(pair_df) >= 3:
                corr, pval = spearmanr(pair_df.iloc[:, 0], pair_df.iloc[:, 1])
                n_trials = len(pair_df)
            else:
                corr, pval = np.nan, np.nan
                n_trials = len(pair_df)

            # Significance label: 1 (positive significant), -1 (negative significant), 0 otherwise
            if pd.notna(pval) and pval < 0.05 and pd.notna(corr):
                significance = 1 if corr > 0 else -1
            else:
                significance = 0

            correlation_rows.append({
                'Patient': patient,
                'Session': session,
                'Radius_pre': radius_pre,
                'Radius_post': radius_post,
                'Pre_Metric': pre_metric,
                'Post_Metric': post_metric,
                'Correlation': corr,
                'Significance': significance,
                'p_value': pval,
                'n_trials': n_trials,
            })

print(f'Processing complete. {processed}/{total_groups} groups processed.')

# Build correlation dataframe
df_all_corr = pd.DataFrame(correlation_rows)

# Sort for readability
patient_num = df_all_corr['Patient'].str.extract(r'(\d+)')[0].astype(float)
session_num = df_all_corr['Session'].str.extract(r'(\d+)')[0].astype(float)
df_all_corr = (
    df_all_corr
    .assign(_patient_num=patient_num, _session_num=session_num)
    .sort_values(
        ['_patient_num', '_session_num', 'Radius_pre', 'Radius_post', 'Pre_Metric', 'Post_Metric'],
        kind='stable'
    )
    .drop(columns=['_patient_num', '_session_num'])
    .reset_index(drop=True)
)

print(f'\ndf_all_corr shape: {df_all_corr.shape}')
print(f'Unique pre-metrics: {df_all_corr["Pre_Metric"].nunique()}')
print(f'Unique post-metrics (targets): {df_all_corr["Post_Metric"].nunique()}')
df_all_corr.head(10)

Processing complete. 318/318 groups processed.

df_all_corr shape: (164088, 10)
Unique pre-metrics: 129
Unique post-metrics (targets): 4


,Patient,Session,Radius_pre,Radius_post,Pre_Metric,Post_Metric,Correlation,Significance,p_value,n_trials
0,P1,S1,100,100,Dimensionality,N1,-0.260402,0,0.104624,40
1,P1,S1,100,100,Dimensionality,N1_t,0.015875,0,0.922547,40
2,P1,S1,100,100,Dimensionality,N2,-0.415951,-1,0.007597,40
3,P1,S1,100,100,Dimensionality,N2_t,-0.213435,0,0.186044,40
4,P1,S1,100,100,Entropy_AEC,N1,-0.296060,0,0.063610,40
5,P1,S1,100,100,Entropy_AEC,N1_t,0.151096,0,0.352025,40
6,P1,S1,100,100,Entropy_AEC,N2,-0.345591,-1,0.028948,40
7,P1,S1,100,100,Entropy_AEC,N2_t,-0.215985,0,0.180715,40
8,P1,S1,100,100,Entropy_COKU,N1,-0.147467,0,0.363842,40
9,P1,S1,100,100,Entropy_COKU,N1_t,-0.067443,0,0.679247,40


# Identify Top Pre-Metrics with Strongest Correlations to N1, N1_t, N2, N2_t

Find which pre-metrics give the strongest correlations (by absolute value) with each target post-metric.

In [10]:
# Filter for target post-metrics and find top pre-metrics
target_metrics = ['N1', 'N1_t', 'N2', 'N2_t']

# Check which targets exist in the data
available_targets = [m for m in target_metrics if m in df_all_corr['Post_Metric'].values]
missing_targets = [m for m in target_metrics if m not in df_all_corr['Post_Metric'].values]

print(f'Target post-metrics available in data: {available_targets}')
if missing_targets:
    print(f'Target post-metrics NOT in data: {missing_targets}')

# Build summary showing top pre-metrics for each target
summary_rows = []

for target_metric in available_targets:
    # Get correlations for this target post-metric
    target_corr = df_all_corr[df_all_corr['Post_Metric'] == target_metric].copy()
    
    # Compute absolute correlation and group by pre-metric
    target_corr['abs_corr'] = target_corr['Correlation'].abs()
    
    # Aggregate across all participant-session groups (mean absolute correlation)
    pre_metric_stats = (
        target_corr
        .groupby('Pre_Metric')
        .agg({
            'Correlation': ['mean', 'std', 'count'],
            'abs_corr': 'mean',
            'Significance': 'mean',
            'p_value': lambda x: (x < 0.05).sum(),
        })
        .round(4)
    )
    
    pre_metric_stats.columns = ['mean_corr', 'std_corr', 'n_groups', 'mean_abs_corr', 'mean_sig', 'n_sig_pval']
    pre_metric_stats = pre_metric_stats.sort_values('mean_abs_corr', ascending=False)
    
    print(f'\n{"="*80}')
    print(f'TOP PRE-METRICS CORRELATING WITH {target_metric}')
    print(f'{"="*80}')
    print(pre_metric_stats.head(20).to_string())
    print(f'\nTotal pre-metrics tested: {len(pre_metric_stats)}')
    
    # Store top 10 for summary
    top_10 = pre_metric_stats.head(10)
    for rank, (pre_metric, row) in enumerate(top_10.iterrows(), 1):
        summary_rows.append({
            'Target_Metric': target_metric,
            'Rank': rank,
            'Pre_Metric': pre_metric,
            'Mean_Abs_Correlation': row['mean_abs_corr'],
            'Mean_Correlation': row['mean_corr'],
            'Std_Correlation': row['std_corr'],
            'N_Groups': int(row['n_groups']),
            'N_Significant': int(row['n_sig_pval']),
        })

# Build summary dataframe
df_top_pre_metrics = pd.DataFrame(summary_rows)

print(f'\n\n{"="*80}')
print(f'SUMMARY: TOP 10 PRE-METRICS FOR EACH TARGET')
print(f'{"="*80}')
print(df_top_pre_metrics.to_string(index=False))

Target post-metrics available in data: ['N1', 'N1_t', 'N2', 'N2_t']

TOP PRE-METRICS CORRELATING WITH N1
                                               mean_corr  std_corr  n_groups  mean_abs_corr  mean_sig  n_sig_pval
Pre_Metric                                                                                                       
N2                                                0.3091    0.2543       318         0.3428    0.4371         143
var_peak_to_peak                                  0.2870    0.2570       318         0.3260    0.4151         138
co-kurtosis_matrix_mean_mean_mat                  0.2846    0.2502       318         0.3260    0.4151         138
co-kurtosis_matrix_avg_clustering_coefficient     0.2839    0.2508       318         0.3231    0.4308         141
var_data                                          0.2814    0.2521       318         0.3204    0.4245         139
mean_square                                       0.2813    0.2521       318         0.3204    0.

# Export Results

Save the computed correlation dataframe and top pre-metrics summary to CSV files.

In [13]:
# Export correlation dataframe with all metrics
output_dir = project_root / 'data' / 'df_results' / 'ieeg_metrics'
output_corr_csv = output_dir / 'df_correlations_allMOIs_100radius_ieeg.csv'
df_all_corr.to_csv(output_corr_csv, index=False)
print(f'✓ Saved all correlations: {output_corr_csv}')
print(f'  Shape: {df_all_corr.shape}')

# Export top pre-metrics summary
output_top_csv = output_dir / 'df_top_preMetrics_for_N1N2_100radius_ieeg.csv'
df_top_pre_metrics.to_csv(output_top_csv, index=False)
print(f'\n✓ Saved top pre-metrics summary: {output_top_csv}')
print(f'  Shape: {df_top_pre_metrics.shape}')

# Show some stats
print(f'\n--- Statistics ---')
print(f'Correlation dataframe:')
print(f'  Total rows: {len(df_all_corr)}')
print(f'  Unique patients: {df_all_corr["Patient"].nunique()}')
print(f'  Unique sessions: {df_all_corr[["Patient", "Session"]].drop_duplicates().shape[0]}')
print(f'  Pre-metrics: {df_all_corr["Pre_Metric"].nunique()}')
print(f'  Post-metrics: {df_all_corr["Post_Metric"].nunique()}')
print(f'\nSignificant (p<0.05) correlations: {(df_all_corr["p_value"] < 0.05).sum()}')

✓ Saved all correlations: /Users/cbc/Documents/GitHub/fufo/notebook/DavideMomi/Revision/State_Dependent_Brain_Stimulation-main/data/df_results/ieeg_metrics/df_correlations_allMOIs_100radius_ieeg.csv
  Shape: (164088, 10)

✓ Saved top pre-metrics summary: /Users/cbc/Documents/GitHub/fufo/notebook/DavideMomi/Revision/State_Dependent_Brain_Stimulation-main/data/df_results/ieeg_metrics/df_top_preMetrics_for_N1N2_100radius_ieeg.csv
  Shape: (40, 8)

--- Statistics ---
Correlation dataframe:
  Total rows: 164088
  Unique patients: 36
  Unique sessions: 318
  Pre-metrics: 129
  Post-metrics: 4

Significant (p<0.05) correlations: 19677


# Optional: Explore Correlations Further

View specific correlation patterns and details.

In [9]:
# Example: View all correlations for a specific target post-metric (e.g., N1)
target = 'N1'

if target in df_all_corr['Post_Metric'].values:
    target_data = df_all_corr[df_all_corr['Post_Metric'] == target].copy()
    target_data['abs_corr'] = target_data['Correlation'].abs()
    target_data_sorted = target_data.sort_values('abs_corr', ascending=False)
    
    print(f'\n{target}: All Pre-Metrics (sorted by |correlation|)')
    print(f'Showing first 30 rows:')
    print(target_data_sorted[['Patient', 'Session', 'Pre_Metric', 'Correlation', 'p_value', 'n_trials']].head(30).to_string(index=False))
else:
    print(f'{target} not found in Post_Metric column')

# Example: View significant correlations only (p < 0.05)
print(f'\n\n--- SIGNIFICANT CORRELATIONS (p < 0.05) ---')
sig_corr = df_all_corr[df_all_corr['p_value'] < 0.05].copy()
print(f'Total significant correlations: {len(sig_corr)} out of {len(df_all_corr)}')

if len(sig_corr) > 0:
    sig_by_target = sig_corr.groupby('Post_Metric').size().sort_values(ascending=False)
    print(f'\nSignificant correlations by Post_Metric:')
    print(sig_by_target)


N1: All Pre-Metrics (sorted by |correlation|)
Showing first 30 rows:
Patient Session                                    Pre_Metric  Correlation      p_value  n_trials
    P25      S1                co-kurtosis_matrix_var_var_mat     0.896429 6.066143e-06        15
    P25      S1                       co-kurtosis_matrix_norm     0.892857 7.494736e-06        15
    P25      S1               co-kurtosis_matrix_mean_var_mat     0.892857 7.494736e-06        15
    P25      S1                                      var_data     0.878571 1.631528e-05        15
    P25      S1                                   mean_square     0.878571 1.631528e-05        15
    P25      S1                                 mean_var_data     0.878571 1.631528e-05        15
    P25      S1                                   Entropy_PCA    -0.871429 2.323648e-05        15
    P18      S6                                            N2     0.870523 4.058586e-10        30
    P25      S1                    co-kurtosis_m

In [12]:
# Display summary: top 10 pre-metrics per target
print("\n=== TOP 10 PRE-METRICS FOR N1, N1_t, N2, N2_t ===\n")

for target in ['N1', 'N1_t', 'N2', 'N2_t']:
    target_data = df_top_pre_metrics[df_top_pre_metrics['Target_Metric'] == target].reset_index(drop=True)
    if len(target_data) > 0:
        print(f"{target}:")
        for _, row in target_data.iterrows():
            print(f"  {int(row['Rank']):2d}. {row['Pre_Metric']:50s} | r={row['Mean_Abs_Correlation']:.4f} | sig={int(row['N_Significant'])}")
        print()


=== TOP 10 PRE-METRICS FOR N1, N1_t, N2, N2_t ===

N1:
   1. N2                                                 | r=0.3428 | sig=143
   2. var_peak_to_peak                                   | r=0.3260 | sig=138
   3. co-kurtosis_matrix_mean_mean_mat                   | r=0.3260 | sig=138
   4. co-kurtosis_matrix_avg_clustering_coefficient      | r=0.3231 | sig=141
   5. var_data                                           | r=0.3204 | sig=139
   6. mean_square                                        | r=0.3204 | sig=139
   7. mean_var_data                                      | r=0.3197 | sig=138
   8. co-kurtosis_matrix_var_mat                         | r=0.3186 | sig=133
   9. co-kurtosis_matrix_var_mean_mat                    | r=0.3184 | sig=128
  10. co-kurtosis_matrix_norm                            | r=0.3121 | sig=128

N1_t:
   1. mean_peak_to_peak                                  | r=0.1726 | sig=22
   2. salience                                           | r=0.1710 | sig=21
   